<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/SVD_Circuits_Demo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVD Circuits Demo

This notebook decomposes one attention head's QK ($W_Q W_K^\top$) and OV ($W_V W_O$) weight maps into orthogonal singular directions. It displays descriptive vocab and activation projections, then measures the effects of retaining or ablating eligible OV directions on one IOI prompt's logit difference. Mathematically distinct directions need not be semantically or causally distinct subfunctions.

Based on Areeb Ahmad, Abhinav Joshi, Ashutosh Modi, "Beyond Components: Singular Vector-Based Interpretability of Transformer Circuits", [arXiv 2511.20273](https://arxiv.org/abs/2511.20273).

## Setup (Ignore)

In [1]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = False
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except ImportError:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    DEVELOPMENT_MODE = True
    from IPython import get_ipython

    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens

Running as a Jupyter notebook - intended for development only!


In [2]:
# NBVAL_IGNORE_OUTPUT
import torch

from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.tools.analysis.svd_circuits import (
    decompose_head,
    patch_along_directions,
    project_activations,
    vocab_readout,
)

MODEL_ID = "gpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerBridge.boot_transformers(MODEL_ID, dtype=torch.float32, device=device)
# The vocab readout projects through the final LayerNorm folded into W_U, so the
# bridge has to be in compatibility mode before any readout is taken.
model.enable_compatibility_mode()
model.eval()
print(f"loaded {MODEL_ID}: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, {device=}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

loaded gpt2: 12 layers, d_model=768, device='cpu'


## 1. Decompose a name-mover head

Layer 9 head 9 is the canonical name-mover head in GPT-2-small's indirect-object-identification (IOI) circuit. Two weight maps describe what it does:

- **QK** $= W_Q W_K^\top$ decides which residual-stream directions the head compares when forming its attention pattern.
- **OV** $= W_V W_O$ decides what the head writes back into the residual stream.

Both are rank at most $d_{\text{head}} = 64$, living inside a $d_{\text{model}} = 768$ space. `FactoredMatrix` computes their SVD without ever materialising the $768 \times 768$ product.

In [3]:
# NBVAL_IGNORE_OUTPUT
# sigma and sigma/sigma_max are float32 reductions, so the fourth decimal can
# differ between CPU kernels. The report is here to show which directions are
# degenerate, not to pin a numeric value.
PROMPT = "When Mary and John went to the store, John gave a drink to"
LAYER, HEAD = 9, 9

decomposition = decompose_head(model, layer=LAYER, head=HEAD, which=("QK", "OV"))
ov = decomposition.OV
assert ov is not None
eligible_ids = [row.idx for row in ov.rank_report if not row.is_degenerate and not row.is_null]
display_ids = eligible_ids[:5]

print(f"{'idx':>4} {'sigma':>10} {'sigma/sigma_max':>16} {'degenerate':>11} {'null':>6}")
for row in ov.rank_report[:12]:
    print(
        f"{row.idx:>4} {row.sigma:>10.4f} {row.sigma_ratio:>16.4f} "
        f"{str(row.is_degenerate):>11} {str(row.is_null):>6}"
    )

 idx      sigma  sigma/sigma_max  degenerate   null
   0     8.4096           1.0000       False  False
   1     8.2600           0.9822       False  False
   2     8.1019           0.9634       False  False
   3     8.0099           0.9525        True  False
   4     7.9312           0.9431        True  False
   5     7.8743           0.9364        True  False
   6     7.8141           0.9292        True  False
   7     7.7938           0.9268        True  False
   8     7.7175           0.9177        True  False
   9     7.6490           0.9096        True  False
  10     7.5680           0.8999        True  False
  11     7.5578           0.8987        True  False


### Reading the rank report

Near-equal consecutive singular values define a subspace whose basis is only defined up to a rotation, so individual directions inside such a block are not attributable. Numerically null directions are arbitrary vectors from the map's null space, and are equally unusable.

`decompose_head` marks degenerate and null directions in the rank report. `logit_signature` refuses individual directions that are degenerate or null. `patch_along_directions` refuses selections that split a degenerate block, but permits complete blocks as subspaces; empty or full-span retained sets require an explicit threshold.

`vocab_readout` and `project_activations` return raw numerical projections, including degenerate columns, without replacing them with block summaries. The displays below therefore select only isolated, non-null directions from the rank report. Without this filtering, "direction 3 is the surname subfunction" would be a claim about an arbitrary basis choice.

## 2. Vocab projections of eligible directions

In [4]:
# NBVAL_IGNORE_OUTPUT
# Token rankings can change across kernels when float32 projection magnitudes nearly tie.
assert all(not ov.rank_report[i].is_degenerate and not ov.rank_report[i].is_null for i in display_ids)
if display_ids:
    # Readout columns retain the original direction IDs, so request the full prefix.
    readout = vocab_readout(model, ov, k=max(display_ids) + 1)
    for i in display_ids:
        top = torch.topk(readout[:, i].abs(), 10).indices
        tokens = [model.tokenizer.decode([int(t)]) for t in top]
        print(f"direction {i}: {tokens}")
else:
    print("No attributable directions to display.")


direction 0: ['Sav', 'AV', 'av', 'adh', 'avour', 'aw', ' Gaw', 'tein', ' Sav', 'amac']
direction 1: ['NI', 'ANI', 'irin', 'Ir', ' NI', 'tro', ' Irwin', 'iq', 'ani', ' Ir']
direction 2: [' Lindsay', ' McKenna', 'etr', ' 339', 'Lind', ' copper', ' seiz', ' Ack', 'gae', ' Instr']
direction 23: [' Sieg', ' Mahjong', ' Buk', 'abal', 'tery', 'Mp', 'jp', 'hap', 'YN', 'ambo']
direction 48: ['û', 'Rand', ' Pupp', ' DN', 'Doc', 'iris', ' Blog', ' Ning', ' Wade', ' Ashe']


Vocab projections can motivate hypotheses but do not establish semantic labels. `SVDInterpreter`'s own docstring warns that singular directions are not reliably interpretable. The next section describes activation coefficients; the intervention section measures changes in the selected prompt's logit difference, without confirming a named subfunction.

## 3. Activation projections on a prompt

In [5]:
# NBVAL_IGNORE_OUTPUT
# The coefficients are float32 reductions over the head's output, so the second
# decimal can differ between CPU kernels (1.75 vs 1.74 on the same input). The
# table is here to show which directions fire at which positions, not to pin a
# numeric value, so its output is not compared.
projection = project_activations(model, ov, PROMPT)

print("prompt tokens:", projection.str_tokens)
print()
assert all(not ov.rank_report[i].is_degenerate and not ov.rank_report[i].is_null for i in display_ids)
if display_ids:
    print(f"{'token':>10} " + " ".join(f"{f'dir{i}':>8}" for i in display_ids))
    for pos, token in enumerate(projection.str_tokens):
        coeffs = " ".join(f"{abs(float(projection.coefficients[pos, i])):>8.2f}" for i in display_ids)
        print(f"{token:>10} {coeffs}")
else:
    print("No attributable directions to display.")


prompt tokens: ['<|endoftext|>', 'When', ' Mary', ' and', ' John', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' a', ' drink', ' to']

     token     dir0     dir1     dir2    dir23    dir48
<|endoftext|>     0.34     0.20     0.17     0.14     0.27
      When     0.34     0.21     0.19     0.12     0.29
      Mary     0.31     0.21     0.19     0.15     0.27
       and     0.20     0.37     0.51     0.35     0.20
      John     0.26     0.19     0.18     0.14     0.30
      went     0.34     0.20     0.17     0.13     0.28
        to     0.27     0.18     0.16     0.15     0.31
       the     0.34     0.22     0.20     0.13     0.27
     store     0.31     0.24     0.23     0.11     0.31
         ,     3.46     0.04     0.65     1.49     0.77
      John     0.27     0.17     0.16     0.10     0.36
      gave     5.59     1.41     2.72     2.42     0.07
         a     0.25     0.27     0.22     0.11     0.29
     drink     0.29     0.19     0.24     0.07     0.32
        t

## 4. Intervention comparisons

`patch_along_directions` projects the head's output onto a chosen singular subspace and measures the resulting change in the selected metric, including downstream computation. It reports:

- `delta_metric`: patched metric minus original metric.
- `baseline_delta_metric`: the mean magnitude over random same-width subspaces drawn inside the head's own OV span, so the control is an arbitrary subspace of *this head's* output rather than an unrelated residual-stream direction.
- `gated`: whether the metric change passed the mode-specific threshold comparison.

The meaning of `gated` depends on the mode:

- With `keep=[i]` (retain only direction $i$), `gated` means `abs(delta_metric) < baseline`: the selected metric changes less than the sampled mean control magnitude.
- With `ablate=[i]` (zero direction $i$), `gated` means `abs(delta_metric) > baseline`: the selected metric changes more than that mean.

`keep=S` and `ablate=complement(S)` resolve to the same retained set, which is why the comparison must depend on the mode. The mean control magnitude is not a confidence bound or p-value; a passing comparison does not establish reconstruction of the head's entire behaviour or statistical separation from arbitrary directions.

The whole-head ablation below provides context for the direction tables. It reports the unmodified metric and the signed change after zeroing the head's output. Its empty-span control coincides with the intervention, so its gate flag is not interpreted.

In [6]:
# NBVAL_IGNORE_OUTPUT
# The deltas are float32 reductions, and a value sitting on a rounding boundary
# can print one digit apart across CPU kernels. The gate's polarity is asserted
# by the test suite; this table is the illustration.
mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")
metric = lambda logits: float(logits[0, -1, mary_token] - logits[0, -1, john_token])

whole_head = patch_along_directions(
    model, ov, PROMPT, metric, keep=[], threshold=0.0, n_baseline=1,
    rng=torch.Generator().manual_seed(0),
)
print(f"original logit difference: {whole_head.original_metric:.3f}")
print(f"whole-head ablation delta: {whole_head.delta_metric:+.3f}")
print()

# Sweep every attributable direction, not just the first few: the gate is a
# comparison against a random control, so which directions pass is the result.
# Deltas print at three decimals because the fourth differs between CPU kernels.
print(f"{'dir':>4} {'sigma':>9} {'delta':>10} {'baseline':>10} {'gated':>6}")
for idx in eligible_ids:
    ov.require_isolated(idx)
    row = ov.rank_report[idx]
    result = patch_along_directions(
        model, ov, PROMPT, metric, keep=[idx], rng=torch.Generator().manual_seed(0)
    )
    print(
        f"{row.idx:>4} {row.sigma:>9.4f} {result.delta_metric:>10.3f} "
        f"{result.baseline_delta_metric:>10.3f} {str(result.gated):>6}"
    )


original logit difference: 3.362
whole-head ablation delta: +0.129

 dir     sigma      delta   baseline  gated


   0    8.4096      0.154      0.127  False


   1    8.2600      0.141      0.127  False


   2    8.1019      0.132      0.127  False


  23    6.8420      0.193      0.127  False


...

  55    5.0464      0.078      0.127   True


  56    4.9738      0.114      0.127   True


  59    4.7041      0.131      0.127  False


  60    4.6103      0.136      0.127  False


  61    4.4351      0.172      0.127  False


In [7]:
# NBVAL_IGNORE_OUTPUT
# Same reason as the keep-mode table: the deltas are float32 reductions, and a
# value sitting on a rounding boundary can print one ulp apart across CPU kernels.
print(f"{'dir':>4} {'sigma':>9} {'delta':>10} {'baseline':>10} {'gated':>6}")
for idx in eligible_ids:
    ov.require_isolated(idx)
    row = ov.rank_report[idx]
    result = patch_along_directions(
        model, ov, PROMPT, metric, ablate=[idx], rng=torch.Generator().manual_seed(0)
    )
    print(
        f"{row.idx:>4} {row.sigma:>9.4f} {result.delta_metric:>10.3f} "
        f"{result.baseline_delta_metric:>10.3f} {str(result.gated):>6}"
    )


 dir     sigma      delta   baseline  gated


   0    8.4096     -0.012      0.041  False


   1    8.2600     -0.015      0.041  False


   2    8.1019     -0.014      0.041  False


  23    6.8420     -0.058      0.041   True


...

  55    5.0464      0.040      0.041  False


  56    4.9738      0.025      0.041  False


  59    4.7041      0.002      0.041  False


  60    4.6103     -0.016      0.041  False


  61    4.4351     -0.032      0.041  False


## 5. What this does and does not show

This example decomposes one GPT-2-small head and measures prompt-specific effects of retaining or ablating eligible OV directions. The reported gate verdicts are comparisons with an averaged random in-span control. They do not establish named subfunctions, statistical separation from arbitrary directions, or replication of the paper's causal subfunction taxonomy.

- **Single head, single model, single prompt.** Everything above concerns layer 9 head 9 of `gpt2-small` on one IOI prompt. Nothing here assembles a multi-head circuit.
- **Descriptive projections.** Orthogonal weight-space directions are mathematically distinct; vocab readouts and activation coefficients do not supply semantic labels.
- **Comparisons, not significance.** The tables report metric changes and mean control magnitudes, not control distributions or confidence bounds. They do not establish that any direction is distinguishable from an arbitrary one.
- **Downstream responses.** The whole-head ablation delta measures the final logit difference after downstream computation, not the head's direct write contribution. A small final-logit effect need not imply a small direct contribution; these measurements do not identify a specific compensating component.
- **Repeatability, not confirmation.** Verdicts near the mean control magnitude can change with the seed or draw count. Fixing the seed makes the comparison repeatable, not scientifically conclusive.
- **No automated labelling.** A passing gate does not name what a direction computes. Named-subfunction claims would need additional prompts, counterfactuals, and suitable control-distribution analyses.